# Auto-Pilot Training — Surface Crack Detection

**Train ResNet50, EfficientNet-B0, ViT-B/16 sequentially with wandb tracking.**

| Step | Model | Est. Time |
|------|-------|-----------|
| 5 | ResNet50 | ~3 min |
| 6 | EfficientNet-B0 | ~4 min |
| 7 | ViT-B/16 | ~8 min |
| 8-9 | Evaluation + Ensemble | ~30 sec |
| 11 | Push to HF Hub | ~30 sec |

In [ ]:
# Cell 1: Clone, path setup, GPU check
import sys, os, subprocess, time, json

PROJECT_DIR = "/content/bootcamp-ace-26-team-7"
GIT_REPO = "esetu-git-public/bootcamp-ace-26-team-7"
GIT_URL = f"https://github.com/{GIT_REPO}.git"

if not os.path.exists(PROJECT_DIR):
    print(f"Cloning {GIT_URL} ...")
    result = subprocess.run(
        ["git", "clone", GIT_URL, PROJECT_DIR],
        capture_output=True, text=True, timeout=120
    )
    if result.returncode != 0:
        print(f"Clone failed:\n{result.stderr}")
        raise SystemExit(1)
    print("Clone complete.")
else:
    print(f"{PROJECT_DIR} already exists — skipping clone")

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f"Working directory: {os.getcwd()}")
print(f"PROJECT_DIR in sys.path: {PROJECT_DIR in sys.path}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be very slow.")
    print("Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Cell 2: Install deps + wandb login (handles interruption gracefully)
print("Installing dependencies...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt",
     "--extra-index-url", "https://download.pytorch.org/whl/cu124"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(f"pip install warnings/errors:\n{result.stderr[-500:]}")
print("pip install complete.")

import wandb
try:
    if not wandb.api.api_key:
        wandb.login()
except Exception:
    print("wandb login skipped (non-interactive or interrupted).")
    print("All other features will work without wandb.")

# Verify critical imports
ok = True
for mod_name in ["torch", "wandb", "PIL", "matplotlib", "numpy", "sklearn", "huggingface_hub"]:
    try:
        __import__(mod_name)
    except ImportError:
        print(f"MISSING: {mod_name}")
        ok = False
if not ok:
    print("Some packages failed to install. Check pip output above.")
else:
    print("All dependencies verified.")

In [ ]:
# Cell 3: Download dataset from HF Hub
from huggingface_hub import hf_hub_download
import zipfile

REPO_ID = "amruthjakku/surface-crack-detection-model"
RAW_DIR = "data/raw"

if os.path.exists(RAW_DIR) and os.listdir(RAW_DIR):
    print(f"Dataset already exists at {RAW_DIR}/ — skipping download")
else:
    try:
        print("Downloading dataset from HF Hub...")
        os.makedirs("data", exist_ok=True)
        zip_path = hf_hub_download(repo_id=REPO_ID, filename="dataset.zip", repo_type="model")
        print(f"Downloaded: {zip_path}")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall("data")
        print(f"Extracted to {RAW_DIR}/")
    except Exception as e:
        print(f"Dataset download failed: {e}")
        print("Upload dataset.zip manually to /content/ and extract it.")
        raise SystemExit(1)

n = len(os.listdir(RAW_DIR)) if os.path.exists(RAW_DIR) else 0
print(f"Raw images: {n}")
if n == 0:
    print("ERROR: No images found. Place images in data/raw/ and re-run.")
    raise SystemExit(1)

In [ ]:
# Cell 4: Stratified split
import src.config as cfg
cfg.Config.WANDB_ENABLED = bool(wandb.api.api_key)

from src.prepare_data import prepare_data
try:
    prepare_data()
    print("Data preparation complete.")
except Exception as e:
    print(f"Data preparation failed: {e}")
    raise SystemExit(1)

from src.dataset import get_dataloaders
train_loader, val_loader, test_loader = get_dataloaders()
print(f"Train: {len(train_loader.dataset)}, Val: {len(val_loader.dataset)}, Test: {len(test_loader.dataset)}")
if len(train_loader.dataset) == 0:
    print("ERROR: Training set is empty.")
    raise SystemExit(1)

In [ ]:
# Cell 5: Train ResNet50
from src.train import run_training

print("=" * 60)
print("Training ResNet50...")
print("=" * 60)
try:
    run_training(model_name="resnet50")
    print("\nResNet50 training complete.")
except Exception as e:
    print(f"\nResNet50 training failed: {e}")
torch.cuda.empty_cache()

In [ ]:
# Cell 6: Train EfficientNet-B0
print("=" * 60)
print("Training EfficientNet-B0...")
print("=" * 60)
try:
    run_training(model_name="efficientnet_b0")
    print("\nEfficientNet-B0 training complete.")
except Exception as e:
    print(f"\nEfficientNet-B0 training failed: {e}")
torch.cuda.empty_cache()

In [ ]:
# Cell 7: Train ViT-B/16
print("=" * 60)
print("Training ViT-B/16...")
print("=" * 60)
try:
    run_training(model_name="vit_b_16")
    print("\nViT-B/16 training complete.")
except Exception as e:
    print(f"\nViT-B/16 training failed: {e}")
torch.cuda.empty_cache()

In [ ]:
# Cell 8: Evaluate each trained model
from src.evaluate import run_evaluation

for model_name in ["resnet50", "efficientnet_b0", "vit_b_16"]:
    path = cfg.Config.get_model_path(model_name)
    if not os.path.exists(path):
        print(f"\nSkipping {model_name} — no checkpoint at {path}")
        continue
    print(f"\n{'=' * 60}")
    print(f"Evaluating {model_name}...")
    print(f"{'=' * 60}")
    try:
        run_evaluation(model_name=model_name)
    except Exception as e:
        print(f"Evaluation failed: {e}")

In [ ]:
# Cell 9: Ensemble evaluation (averages softmax across models)
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import Config
from src.dataset import get_dataloaders
from src.model import get_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
models_to_load = [m for m in Config.ENSEMBLE_MODELS if os.path.exists(Config.get_model_path(m))]

if len(models_to_load) < 2:
    print(f"Ensemble skipped — need >= 2 trained models, have {len(models_to_load)}")
else:
    ensemble_nets = []
    for name in models_to_load:
        net = get_model(model_name=name, num_classes=Config.NUM_CLASSES, pretrained=False)
        net.load_state_dict(torch.load(Config.get_model_path(name), map_location=device))
        net = net.to(device).eval()
        ensemble_nets.append(net)
        print(f"Loaded {name}")

    _, _, test_loader = get_dataloaders()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            probs = torch.zeros(images.size(0), Config.NUM_CLASSES).to(device)
            for net in ensemble_nets:
                probs += torch.softmax(net(images), dim=1)
            probs /= len(ensemble_nets)
            y_true.extend(labels.numpy())
            y_pred.extend(torch.max(probs, 1)[1].cpu().numpy())

    y_true, y_pred = np.array(y_true), np.array(y_pred)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    report = classification_report(y_true, y_pred, target_names=Config.CLASSES, zero_division=0)

    print(f"\n{'=' * 60}")
    print(f"Ensemble ({'+'.join(models_to_load)}) Results")
    print(f"{'=' * 60}")
    print(f"Test Accuracy: {acc:.4f}")
    print(f"Test Weighted F1: {f1:.4f}")
    print(f"\nClassification Report:\n{report}")

    if Config.WANDB_ENABLED:
        ensemble_name = "ensemble-" + "+".join(models_to_load)
        wandb.init(
            project=Config.WANDB_PROJECT,
            entity=Config.WANDB_ENTITY,
            name=f"eval-{ensemble_name}-{time.strftime('%Y%m%d-%H%M%S')}",
            config={"model": ensemble_name}
        )
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=Config.CLASSES, yticklabels=Config.CLASSES)
        plt.title(f"Confusion Matrix — {ensemble_name}")
        cm_path = os.path.join(Config.REPORTS_DIR, f"confusion_matrix_{ensemble_name}.png")
        os.makedirs(Config.REPORTS_DIR, exist_ok=True)
        plt.savefig(cm_path); plt.close()
        wandb.log({
            "test_accuracy": acc,
            "test_weighted_f1": f1,
            "confusion_matrix": wandb.Image(cm_path)
        })
        wandb.finish()

In [ ]:
# Cell 10: Summary — wandb link + local paths
entity = Config.WANDB_ENTITY or (wandb.api.default_entity if wandb.api.api_key else "<your-entity>")
print("=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)
if wandb.api.api_key:
    print(f"wandb: https://wandb.ai/{entity}/{Config.WANDB_PROJECT}")
else:
    print("wandb: not logged in (no dashboard)")
print()
for name in ["resnet50", "efficientnet_b0", "vit_b_16"]:
    path = Config.get_model_path(name)
    status = f"{os.path.getsize(path)/1024/1024:.1f} MB" if os.path.exists(path) else "not trained"
    print(f"  {name:20s} {status}")
print(f"  {'Ensemble':20s} {Config.ENSEMBLE_MODELS}")

from src.visualize import plot_session_comparison
from IPython.display import display, Image

viz_path = plot_session_comparison()
if viz_path:
    display(Image(viz_path))
else:
    print("Not enough sessions to plot comparison.")

In [ ]:
# Cell 10b: Save session to history
import sys, os
from src.session import SessionTracker
from src.config import Config

results = {}
for name in ["resnet50", "efficientnet_b0", "vit_b_16"]:
    path = Config.get_model_path(name)
    if os.path.exists(path):
        results[name] = SessionTracker.build_entry(name, status="ok")
    else:
        results[name] = {"status": "skipped"}

SessionTracker.new_session(device=Config.DEVICE, mode="full", model_results=results)
print("Session saved.")

In [ ]:
# Cell 11: Push trained models to HF Hub
HF_TOKEN = input("Enter HF token (write access) or press Enter to skip: ").strip()

if not HF_TOKEN:
    print("Skipping push.")
else:
    from huggingface_hub import HfApi, login
    try:
        login(token=HF_TOKEN)
        api = HfApi()
        pushed = 0

        for model_name in ["resnet50", "efficientnet_b0", "vit_b_16"]:
            path = Config.get_model_path(model_name)
            if not os.path.exists(path):
                continue
            try:
                api.upload_file(
                    path_or_fileobj=path,
                    path_in_repo=f"{model_name}_best.pth",
                    repo_id=REPO_ID, repo_type="model",
                )
                print(f"Pushed {model_name}_best.pth")
                pushed += 1
            except Exception as e:
                print(f"Failed to push {model_name}: {e}")

        if os.path.isdir(Config.REPORTS_DIR):
            for fname in os.listdir(Config.REPORTS_DIR):
                fpath = os.path.join(Config.REPORTS_DIR, fname)
                if not os.path.isfile(fpath):
                    continue
                try:
                    api.upload_file(
                        path_or_fileobj=fpath,
                        path_in_repo=f"reports/{fname}",
                        repo_id=REPO_ID, repo_type="model",
                    )
                    print(f"Pushed reports/{fname}")
                except Exception:
                    pass

        print(f"\nPushed {pushed} model(s) to {REPO_ID}")
    except Exception as e:
        print(f"Push failed: {e}")

In [ ]:
# Cell 12: Open wandb dashboard
if wandb.api.api_key:
    entity = Config.WANDB_ENTITY or wandb.api.default_entity
    url = f"https://wandb.ai/{entity}/{Config.WANDB_PROJECT}"
    print(f"Dashboard: {url}")
    from IPython.display import display, HTML
    display(HTML(f'<a href="{url}" target="_blank">Open wandb Dashboard</a>'))
else:
    print("wandb not logged in — no dashboard available.")

print("\nAuto-pilot training complete!")